# INSTALL / IMPORT LIBRARIES

In [0]:
%pip install simple-salesforce

In [0]:
import json
import os

config_path ='/Workspace/Users/kortum.facturas@gmail.com/Drafts/GAMBILL PROJECTS/3RD PROJECT/config.json'
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)

shopify_name = config.get("SHOPIFY_STORE")
shopify_name_ver = '2026-04'
shopify_token = config.get("X-Shopify-Access-Token")

salesforce_name = config.get("SALESFORCE")
salesforce_key = config.get("SALESFORCE_API_KEY")
salesforce_secret_key = config.get("SALESFORCE_API_KEY_SECRET")
salesforce_user = config.get("SALESFORCE_USER")
salesforce_pass = config.get("SALESFORCE_PASS")
salesforce_tok = config.get("SALESFORCE_TOKEN")

In [0]:
import requests
import datetime
import time
import logging
from pyspark.sql import Window
from pyspark.sql import Row
from delta.tables import DeltaTable
from pyspark.sql import functions as F


# BRONZE LAYER / API REQUEST FROM THE 3 DIFFERENT PLATFORMS

In [0]:

zendesk_volume_path ='/Volumes/bronze/customer_master_data_management/zendesk_customers'
df_zendesk = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(zendesk_volume_path)
df_zendesk.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('bronze.customer_master_data_management.zendesk_customers_table')
display(df_zendesk)

In [0]:
url = f"https://{shopify_name}.myshopify.com/admin/api/{shopify_name_ver}/customers.json"
headers = {
    "X-Shopify-Access-Token": shopify_token,
    "Content-Type": "application/json"
}

LIMIT = 250
customers_data = []
# params = {"limit": LIMIT}
params = {"limit": LIMIT,
          "fields": "id, first_name, last_name, phone, email, default_address"}
has_next_page = True

while has_next_page:
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        customers = data.get("customers", [])
        customers_data.extend(customers)
        # print(customers_data)
        # print(response.headers)
        # 1. Capture the Link header
        link_header = response.headers.get("Link")
        next_url = None
        
        # 2. Check if a link header exists
        if link_header:
            # Shopify headers look like: <URL>; rel="previous", <URL>; rel="next"
            # Split by commas to evaluate links individually
            links = link_header.split(",")
            for link in links:
                if 'rel="next"' in link:
                    # Isolate just the URL inside the angle brackets <>
                    next_url = link.split(";")[0].strip("<> ")
        
        # 3. Control loop continuation cleanly
        if next_url:
            url = next_url
            params = {}  # Keep empty; next_url has built-in page_info
        else:
            has_next_page = False  # Safely breaks loop when no next page exists
            
    else:
        print(f"Error fetching data: {response.status_code}")
        break

# 1. Blueprint matching ONLY the requested fields
from pyspark.sql.types import StructType, StructField, LongType, StringType
shopify_schema = StructType([
    StructField("id", LongType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("default_address", StructType([
        StructField("city", StringType(), True),
        StructField("province_code", StringType(), True),
        StructField("zip", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("address1", StringType(), True)
    ]), True)
])

# 2. Convert to distributed Spark DataFrame safely
df_raw = spark.createDataFrame(customers_data, schema=shopify_schema)

# 3. Clean and flatten the columns cleanly
df_flat = df_raw.select(
    F.col("id").alias("customer_id"),
    F.col("first_name"),
    F.col("last_name"),
    F.col("email"),
    F.col("default_address.city").alias("city"),
    F.col("default_address.province_code").alias("state"),
    F.col("default_address.zip").alias("zip"),
    F.col("default_address.phone").alias("phone"),
    F.col("default_address.address1").alias("address")
)

df_flat.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('bronze.customer_master_data_management.shopify_customers_table')
display(df_flat)

In [0]:

# [{'id': 10369863942328, 
#   'created_at': '2026-06-19T19:11:53-04:00', 
#   'updated_at': '2026-06-19T19:11:53-04:00', 
#   'first_name': 'Jason', 
#   'last_name': 'Andrade', 
#   'orders_count': 0, 
#   'state': 'disabled', 
#   'total_spent': '0.00', 
#   'last_order_id': None, 
#   'note': None, 
#   'verified_email': True, 
#   'multipass_identifier': None, 
#   'tax_exempt': False, 
#   'tags': '', 
#   'last_order_name': None, 
#   'email': 'jason.andrade@gmail.com', 
#   'phone': None, 
#   'currency': 'USD', 
#   'addresses': [{'id': 12235531092152, 'customer_id': 10369863942328, 'first_name': 'Jason', 'last_name': 'Andrade', 
#                  'company': None,   'address1': None, 'address2': None, 'city': 'Burketown', 'province': None, 'country': None, 'zip': '35552', 'phone': None, 'name': 'Jason Andrade', 'province_code': None, 'country_code': None, 'country_name': None, 'default': True}], 
#   'tax_exemptions': [], 
#   'email_marketing_consent': {'state': 'not_subscribed', 'opt_in_level': 'single_opt_in', 'consent_updated_at': None}, 'sms_marketing_consent': None, 
#   'admin_graphql_api_id': 'gid://shopify/Customer/10369863942328', 
#   'default_address': {'id': 12235531092152, 'customer_id': 10369863942328, 'first_name': 'Jason', 'last_name': 'Andrade', 
#                       'company': None, 'address1': None, 'address2': None, 'city': 'Burketown', 'province': None, 'country': None, 'zip': '35552', 'phone': None, 'name': 'Jason Andrade', 'province_code': None, 'country_code': None, 'country_name': None, 'default': True}}]

In [0]:
from simple_salesforce import Salesforce
from pyspark.sql.types import StructType, StructField, StringType

# Define credentials 
USER = salesforce_user
PWD = salesforce_pass
TOK = salesforce_tok  
KEY = salesforce_key
SEC = salesforce_secret_key

print("Initializing Secure Connection to Salesforce Production/Developer Instance...")

try:
    sf = Salesforce(
        username=USER,
        password=PWD,
        security_token=TOK,
        consumer_key=KEY,
        consumer_secret=SEC,
        domain="login"
    )
    print("SUCCESS: Successfully authenticated to Production!")

    print("Extracting ALL unconverted lead data via API query...")
    
    # FIX: Changed 'query' to 'query_all' to automatically paginate beyond the 2,000 row limit
    leads = sf.query_all("SELECT sfdc_id__c, FirstName, LastName, City, State, Phone, Email FROM Lead ")
    
    if leads['records']:
        # Format rows cleanly, dropping Salesforce system metadata attributes
        cleaned_records = [{k: v for k, v in row.items() if k != 'attributes'} for row in leads['records']]
        
        # Explicit Schema declaration guarantees data types match inside Databricks
        schema = StructType([
            StructField("sfdc_id__c", StringType(), True),
            StructField("FirstName", StringType(), True),
            StructField("LastName", StringType(), True),
            StructField("City", StringType(), True),
            StructField("State", StringType(), True),
            StructField("Phone", StringType(), True),
            StructField("Email", StringType(), True)
        ])
        
        # Parallelize the full paginated dataset directly into the Databricks Spark engine
        spark_df = spark.createDataFrame(cleaned_records, schema=schema)
        
        # Verify the record count in your console output
        print(f"Successfully processed {spark_df.count()} leads into Databricks Spark Engine.")
        
        # Output interactive results grid inside your notebook
        display(spark_df)
    else:
        print("API Connection successful, but zero unconverted leads matched your criteria.")

except Exception as e:
    raise Exception(f"Production Pipeline Pipeline Terminated: {str(e)}")

spark_df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('bronze.customer_master_data_management.salesforce_leads_table')